In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, d1fb2d1b-41b1-4e0f-9a7e-19822d592648, 3, Finished, Available, Finished, False)

<class 'pyspark.sql.session.SparkSession'>


**Read Customers.csv file from souce and store to a data frame**

In [ ]:
customers_path = "Files/Bronze/customers.csv"
customers_df = spark.read.option("header", True)\
                         .option("inferSchema", True)\
                         .option("multiLine", True)\
                         .option("quote","\"")\
                         .option("escape", "\"")\
                         .csv(customers_path)

customers_df.show(5)

Validating if the Schema Inference Worked Correctly and counting the number of records

In [ ]:
customers_df.printSchema()
customers_df.count()

Write Bronze Delta Table

Now we persist the dataframe into the Lakehouse Bronze layer.

In [5]:
customers_df.write.format("delta")\
.mode("overwrite")\
.saveAsTable("bronze_customers")

StatementMeta(, f9591fe0-9eda-41ac-8ada-d32950074c96, 7, Finished, Available, Finished, False)

verified count of records using SQL

In [6]:
spark.sql("SELECT COUNT(*) FROM bronze_customers").show()

StatementMeta(, f9591fe0-9eda-41ac-8ada-d32950074c96, 8, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
| 1000000|
+--------+



** **Need to read-> create DF-> Write as Delta Table** **
<mark>bronze_order_items
bronze_payments
bronze_products
bronze_customers_cdc</mark>

In [1]:
orders_path = "Files/Bronze/orders.parquet"
orders_df = spark.read.option("header",True)\
                      .option("inferSchema", True)\
                      .option("multiline", True)\
                      .option("quote", "\"")\
                      .option("escape", "\"")\
                      .parquet(orders_path)

orders_df.printSchema()
orders_df.show(5)
orders_df.count()


StatementMeta(, 67427e1e-96cf-4bc9-96d4-316dff7942a5, 3, Finished, Available, Finished, False)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_value: double (nullable = true)

+--------+-----------+----------+------------+-----------+
|order_id|customer_id|order_date|order_status|order_value|
+--------+-----------+----------+------------+-----------+
|       1|     805835|2024-03-29|      PLACED|    1743.32|
|       2|     419734|2024-11-21|      PLACED|    1899.51|
|       3|     948279|2024-03-19|   CANCELLED|    1328.48|
|       4|     664708|2025-03-18|   DELIVERED|     890.72|
|       5|     983360|2025-04-30|   CANCELLED|     331.65|
+--------+-----------+----------+------------+-----------+
only showing top 5 rows



8000000

In [2]:
orders_df.write.format("delta")\
               .mode("overwrite")\
               .saveAsTable("bronze_orders")

StatementMeta(, 67427e1e-96cf-4bc9-96d4-316dff7942a5, 4, Finished, Available, Finished, False)

**Read order_items from source save as df and write to Delta table**

In [13]:
order_items_path = "Files/Bronze/order_items.parquet"

order_items_df = spark.read\
                      .parquet(order_items_path)
                      

order_items_df.printSchema()
order_items_df.count()
order_items_df.show(5)

StatementMeta(, 67427e1e-96cf-4bc9-96d4-316dff7942a5, 15, Finished, Available, Finished, False)

root
 |-- item_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- item_price: double (nullable = true)

+-------+--------+----------+--------+----------+
|item_id|order_id|product_id|quantity|item_price|
+-------+--------+----------+--------+----------+
|      1|       1|     12322|       3|    342.41|
|      2|       1|     43051|       4|    489.08|
|      3|       1|     19188|       4|    495.35|
|      4|       2|     30153|       5|     55.37|
|      5|       2|     12841|       4|    197.53|
+-------+--------+----------+--------+----------+
only showing top 5 rows



In [14]:
order_items_df.write.format("delta")\
              .mode("overwrite")\
              .saveAsTable("bronze_order_items")

StatementMeta(, 67427e1e-96cf-4bc9-96d4-316dff7942a5, 16, Finished, Available, Finished, False)

In [16]:
products_df_path = "Files/Bronze/products.parquet"

products_df = spark.read.parquet(products_df_path)

products_df.printSchema()
products_df.count()
products_df.show(5)

StatementMeta(, 67427e1e-96cf-4bc9-96d4-316dff7942a5, 18, Finished, Available, Finished, False)

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- product_description: string (nullable = true)

+----------+------------+-----------+------+------+--------------------+
|product_id|product_name|   category| price|  cost| product_description|
+----------+------------+-----------+------+------+--------------------+
|         1|       could|Electronics|225.71| 87.76|Ten during wish. ...|
|         2|        also|      Books|110.78| 92.07|Up account single...|
|         3|   religious|   Clothing|373.55|160.51|Value thought beh...|
|         4|    security|   Clothing|417.77| 100.0|Must left fly for...|
|         5| environment|     Beauty|235.11|173.25|Be customer his m...|
+----------+------------+-----------+------+------+--------------------+
only showing top 5 rows



In [17]:
products_df.write.format("delta")\
           .mode("overwrite")\
           .saveAsTable("bronze_products")

StatementMeta(, 67427e1e-96cf-4bc9-96d4-316dff7942a5, 19, Finished, Available, Finished, False)

In [20]:
customers_cdc_updates_path = "Files/Bronze/customers_cdc_updates.csv"

customers_cdc_updates_df = spark.read.option("header",True)\
                                     .option("inferSchema", True)\
                                     .option("multiline",True)\
                                     .option("quote","\"")\
                                     .option("escape","\"")\
                                     .csv(customers_cdc_updates_path)

customers_cdc_updates_df.printSchema()
customers_cdc_updates_df.count()
customers_cdc_updates_df.show(5)


StatementMeta(, 67427e1e-96cf-4bc9-96d4-316dff7942a5, 22, Finished, Available, Finished, False)

root
 |-- customer_id: integer (nullable = true)
 |-- operation: string (nullable = true)
 |-- update_time: timestamp (nullable = true)

+-----------+---------+-------------------+
|customer_id|operation|        update_time|
+-----------+---------+-------------------+
|      97510|   DELETE|2026-02-16 10:33:22|
|     406863|   DELETE|2026-01-24 09:45:21|
|     641505|   INSERT|2026-01-01 17:52:11|
|     623036|   INSERT|2026-02-22 19:53:57|
|     624613|   INSERT|2026-01-14 15:30:17|
+-----------+---------+-------------------+
only showing top 5 rows



In [22]:
customers_cdc_updates_df.write.format("delta")\
                              .mode("overwrite")\
                              .saveAsTable("bronze_customer_cdc_updates")

StatementMeta(, 67427e1e-96cf-4bc9-96d4-316dff7942a5, 24, Finished, Available, Finished, False)

In [23]:
payments_df_path = "Files/Bronze/payments.csv"

payments_df = spark.read.option("header",True)\
                        .option("inferSchema", True)\
                        .option("quote","\"")\
                        .option("escape","\"")\
                        .csv(payments_df_path)

payments_df.printSchema()
payments_df.count()
payments_df.show(5)

StatementMeta(, 67427e1e-96cf-4bc9-96d4-316dff7942a5, 25, Finished, Available, Finished, False)

root
 |-- payment_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_amount: double (nullable = true)

+----------+--------+--------------+--------------+--------------+
|payment_id|order_id|payment_method|payment_status|payment_amount|
+----------+--------+--------------+--------------+--------------+
|         1|       1|        WALLET|       SUCCESS|        406.73|
|         2|       2|           UPI|       SUCCESS|        639.89|
|         3|       3|          CARD|       SUCCESS|        1037.3|
|         4|       4|        WALLET|        FAILED|       1108.74|
|         5|       5|           UPI|       SUCCESS|       1437.91|
+----------+--------+--------------+--------------+--------------+
only showing top 5 rows



In [24]:
payments_df.write.format("delta")\
                 .mode("overwrite")\
                 .saveAsTable("bronze_payments")

StatementMeta(, 67427e1e-96cf-4bc9-96d4-316dff7942a5, 26, Finished, Available, Finished, False)

In [1]:
spark.sql("DESCRIBE HISTORY bronze_customers").show()

StatementMeta(, 91b50634-e570-4112-84ae-735e11315489, 3, Finished, Available, Finished, False)

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      0|2026-03-08 09:16:...|  NULL|    NULL|CREATE OR REPLACE...|{partitionBy -> [...|NULL|    NULL|     NULL|       NULL|  Serializable|        false|{numFiles -> 1, n...|        NULL|Apache-Spark/3.5....|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+-----------